In [16]:
from dotenv import load_dotenv
import os

load_dotenv()  # Loads .env file in current directory


True

In [17]:
import os

# Load from secure place or prompt the user
api_key = os.getenv("LANGEXTRACT_API_KEY")
if not api_key:
    raise RuntimeError("Please set LANGEXTRACT_API_KEY in the environment before running.")

# Ensure it's available to the library
os.environ["LANGEXTRACT_API_KEY"] = api_key

In [18]:
import langextract as lx
import textwrap

# 1. Define the prompt and extraction rules
prompt = textwrap.dedent("""\
    Extract characters, emotions, and relationships in order of appearance.
    Use exact text for extractions. Do not paraphrase or overlap entities.
    Provide meaningful attributes for each entity to add context.""")

# 2. Provide a high-quality example to guide the model
examples = [
    lx.data.ExampleData(
        text="ROMEO. But soft! What light through yonder window breaks? It is the east, and Juliet is the sun.",
        extractions=[
            lx.data.Extraction(
                extraction_class="character",
                extraction_text="ROMEO",
                attributes={"emotional_state": "wonder"}
            ),
            lx.data.Extraction(
                extraction_class="emotion",
                extraction_text="But soft!",
                attributes={"feeling": "gentle awe"}
            ),
            lx.data.Extraction(
                extraction_class="relationship",
                extraction_text="Juliet is the sun",
                attributes={"type": "metaphor"}
            ),
        ]
    )
]

In [19]:
# import os
# LANGEXTRACT_API_KEY=os.getenv("LANGEXTRACT_API_KEY")
# result = lx.extract(
#     text_or_documents="https://www.gutenberg.org/files/1513/1513-0.txt",
#     prompt_description=prompt,
#     examples=examples,
#     model_id="gemini-2.5-flash",
#     extraction_passes=3,    # Improves recall through multiple passes
#     max_workers=20,         # Parallel processing for speed
#     max_char_buffer=1000    # Smaller contexts for better accuracy
# )
import requests

url = "https://www.gutenberg.org/files/1513/1513-0.txt"
text = requests.get(url).text

short_text = text[:3000]
result = lx.extract(
    text_or_documents=short_text,  # trimmed input
    prompt_description=prompt,
    examples=examples,
    model_id="gemini-2.5-flash",
    extraction_passes=1,
    max_workers=1,
    max_char_buffer=4000
)

LangExtract: model=gemini-2.5-flash, current=3,000 chars, processed=0 chars:  [03:12]


In [20]:
print(f"Extracted {len(result.extractions)} entities from {len(result.text):,} characters")

Extracted 75 entities from 3,000 characters


In [22]:
from collections import Counter, defaultdict
# Analyze character mentions
characters = {}
for e in result.extractions:
    if e.extraction_class == "character":
        char_name = e.extraction_text
        if char_name not in characters:
            characters[char_name] = {"count": 0, "attributes": set()}
        characters[char_name]["count"] += 1
        if e.attributes:
            for attr_key, attr_val in e.attributes.items():
                characters[char_name]["attributes"].add(f"{attr_key}: {attr_val}")

# Print character summary
print(f"\nCHARACTER SUMMARY ({len(characters)} unique characters)")
print("=" * 60)

sorted_chars = sorted(characters.items(), key=lambda x: x[1]["count"], reverse=True)
for char_name, char_data in sorted_chars[:10]:  # Top 10 characters
    attrs_preview = list(char_data["attributes"])[:3]
    attrs_str = f" ({', '.join(attrs_preview)})" if attrs_preview else ""
    print(f"{char_name}: {char_data['count']} mentions{attrs_str}")

# Entity type breakdown
entity_counts = Counter(e.extraction_class for e in result.extractions)
print(f"\nENTITY TYPE BREAKDOWN")
print("=" * 60)
for entity_type, count in entity_counts.most_common():
    percentage = (count / len(result.extractions)) * 100
    print(f"{entity_type}: {count} ({percentage:.1f}%)")


CHARACTER SUMMARY (35 unique characters)
SAMPSON: 3 mentions (emotional_state: defiant, emotional_state: belligerent)
GREGORY: 2 mentions (emotional_state: witty)
Gregory: 2 mentions
ESCALUS: 1 mentions
MERCUTIO: 1 mentions
PARIS: 1 mentions
Page: 1 mentions
MONTAGUE: 1 mentions
LADY MONTAGUE: 1 mentions
ROMEO: 1 mentions

ENTITY TYPE BREAKDOWN
character: 39 (52.0%)
relationship: 32 (42.7%)
emotion: 4 (5.3%)


In [23]:
import os
import asyncio
from langchain_community.document_loaders import PyPDFLoader

source_folder = "C:/Users/KPMG/Downloads/Promotion-PDFs"
async def load_all_pdfs(folder_path):
    pages = []
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            file_path = os.path.join(folder_path, filename)
            loader = PyPDFLoader(file_path)
            async for page in loader.alazy_load():
                # add filename as metadata so you know source PDF later
                page.metadata["source_filename"] = filename
                pages.append(page)
    return pages

# In a Jupyter cell:
pages = await load_all_pdfs(source_folder)
print(f"Loaded {len(pages)} pages")

Loaded 4 pages


In [31]:
pdf_text=""
for doc in pages:
    pdf_text += doc.page_content
print(pdf_text)

Exp X
Simple Promotion 
Start Date: 2025-04-17
End Date: 2025-05-31
Promotion ID Component ID Item ID Discount Type Discount Value
PROMO422 COMP422 ITEM001 % Off 30Exp X
Simple Promotion 
Start Date: 2025-12-01
End Date: 2025-12-31
Promotion ID Component ID Item ID Discount Type Discount Value
PROMO432 COMP432 ITEM009 % Off 50
PROMO432 COMP432 ITEM029 % Off 50
PROMO432 COMP432 ITEM049 % Off 50Exp X
Simple Promotion 
Start Date: 2025-06-17
End Date: 2025-06-30
Promotion ID Component ID Item ID Discount Type Discount Value
PROMO433 COMP433 ITEM001 % Off 30
PROMO433 COMP433 ITEM021 % Off 30
PROMO433 COMP433 ITEM041 % Off 30Exp X
Simple Promotion 
Start Date: 2025-11-01
End Date: 2025-11-15
Promotion ID Component ID Item ID Discount Type Discount Value
PROMO448 COMP448 ITEM002 % Off 31
PROMO448 COMP448 ITEM005 % Off 31
PROMO448 COMP448 ITEM022 % Off 31
PROMO448 COMP448 ITEM025 % Off 31
PROMO448 COMP448 ITEM042 % Off 31
PROMO448 COMP448 ITEM045 % Off 31


In [41]:
prompt = """
Extract the following from the document:
- Promotion Type
- Start Date
- End Date
- Promotion ID
- Component ID
- List of Item IDs
- Discount Type
- Discount Value

Use exact spans. Do not paraphrase.
"""
# examples = [
#     lx.data.ExampleData(
#         text=textwrap.dedent("""\
#             Simple Promotion
#             Start Date: 2024-01-01
#             End Date: 2024-01-10
#             Promotion ID Component ID Item ID Discount Type Discount Value
#             PROMO123 COMP123 ITEM001 % Off 20
#             PROMO123 COMP123 ITEM002 % Off 20
#         """),
#         extractions=[
#             lx.data.Extraction(
#                 extraction_class="promotion_type",
#                 extraction_text="Simple Promotion"
#             ),
#             lx.data.Extraction(
#                 extraction_class="start_date",
#                 extraction_text="2024-01-01"
#             ),
#             lx.data.Extraction(
#                 extraction_class="end_date",
#                 extraction_text="2024-01-10"
#             ),
#             lx.data.Extraction(
#                 extraction_class="promotion_id",
#                 extraction_text="PROMO123"
#             ),
#             lx.data.Extraction(
#                 extraction_class="component_id",
#                 extraction_text="COMP123"
#             ),
#             lx.data.Extraction(
#                 extraction_class="discount_type",
#                 extraction_text="% Off"
#             ),
#             lx.data.Extraction(
#                 extraction_class="discount_value",
#                 extraction_text="20"
#             ),
#             lx.data.Extraction(
#                 extraction_class="item_id",
#                 extraction_text="ITEM001"
#             ),
#             lx.data.Extraction(
#                 extraction_class="item_id",
#                 extraction_text="ITEM002"
#             ),
#         ]
#     )
# ]
lx.data.Extraction(
    extraction_class="promotion",
    extraction_text="PROMO123",
    attributes={
        "type": "Simple Promotion",
        "start_date": "2024-01-01",
        "end_date": "2024-01-10",
        "component_id": "COMP123",
        "discount_type": "% Off",
        "discount_value": "20"
    }
),
lx.data.Extraction(
    extraction_class="item",
    extraction_text="ITEM001",
    attributes={"promotion_id": "PROMO123"}
),
lx.data.Extraction(
    extraction_class="item",
    extraction_text="ITEM002",
    attributes={"promotion_id": "PROMO123"}
)

Extraction(extraction_class='item', extraction_text='ITEM002', char_interval=None, alignment_status=None, extraction_index=None, group_index=None, description=None, attributes={'promotion_id': 'PROMO123'})

In [42]:
import langextract as lx

result = lx.extract(
    text_or_documents=pdf_text[:5000],  # limit for testing
    prompt_description=prompt,
    examples=examples,
    model_id="gemini-2.5-flash",
    extraction_passes=1,
    max_workers=1,
    max_char_buffer=4000
)

LangExtract: model=gemini-2.5-flash, current=962 chars, processed=0 chars:  [00:20]


In [43]:
print(f"Extracted {len(result.extractions)} entities from {len(result.text):,} characters")

Extracted 41 entities from 962 characters


In [38]:
from collections import Counter, defaultdict
# Analyze character mentions
characters = {}
for e in result.extractions:
    if e.extraction_class == "character":
        char_name = e.extraction_text
        if char_name not in characters:
            characters[char_name] = {"count": 0, "attributes": set()}
        characters[char_name]["count"] += 1
        if e.attributes:
            for attr_key, attr_val in e.attributes.items():
                characters[char_name]["attributes"].add(f"{attr_key}: {attr_val}")

# Print character summary
print(f"\nCHARACTER SUMMARY ({len(characters)} unique characters)")
print("=" * 60)

sorted_chars = sorted(characters.items(), key=lambda x: x[1]["count"], reverse=True)
for char_name, char_data in sorted_chars[:10]:  # Top 10 characters
    attrs_preview = list(char_data["attributes"])[:3]
    attrs_str = f" ({', '.join(attrs_preview)})" if attrs_preview else ""
    print(f"{char_name}: {char_data['count']} mentions{attrs_str}")

# Entity type breakdown
entity_counts = Counter(e.extraction_class for e in result.extractions)
print(f"\nENTITY TYPE BREAKDOWN")
print("=" * 60)
for entity_type, count in entity_counts.most_common():
    percentage = (count / len(result.extractions)) * 100
    print(f"{entity_type}: {count} ({percentage:.1f}%)")


CHARACTER SUMMARY (0 unique characters)

ENTITY TYPE BREAKDOWN
item_id: 13 (31.7%)
promotion_type: 4 (9.8%)
start_date: 4 (9.8%)
end_date: 4 (9.8%)
promotion_id: 4 (9.8%)
component_id: 4 (9.8%)
discount_type: 4 (9.8%)
discount_value: 4 (9.8%)
